# ⛓️ LCEL — LangChain Expression Language

## What is LCEL?
LCEL is LangChain's **composition syntax** using the pipe `|` operator.

```python
chain = prompt | llm | parser
result = chain.invoke({"topic": "AI"})
```

## Mental Model: Linux Pipes
Linux: `cat file.txt | grep error | wc -l`
- `cat` outputs text
- `grep` filters it
- `wc` counts lines

LCEL: `prompt | llm | parser`
- `prompt` outputs Messages
- `llm` processes Messages → AIMessage
- `parser` extracts AIMessage → string

## Why LCEL Exists
Before LCEL, you'd write:
```python
prompt_output = prompt.format(**inputs)
llm_output = llm.invoke(prompt_output)
final = parser.parse(llm_output)
```

LCEL makes this declarative and gives you:
- Streaming for free
- Async for free
- Batching for free
- Tracing for free
- Retries for free


In [ ]:
# ── How | Works: The __or__ Method ─────────────────────────────────────
# WHY the pipe operator?
# In Python, | is the __or__ method. LangChain overrides it.
# When you write prompt | llm, Python calls prompt.__or__(llm)
# This returns a RunnableSequence object.

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

prompt = ChatPromptTemplate.from_template('Tell me about {topic}')
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
parser = StrOutputParser()

chain = prompt | llm | parser

print('Chain type:', type(chain).__name__)  # RunnableSequence!
print('Steps:', [type(s).__name__ for s in chain.steps])

In [ ]:
# ── invoke vs stream vs batch ───────────────────────────────────────────
# ALL Runnables support these 3 execution modes.
# This is LCEL's killer feature — write once, run in any mode.

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template('Explain {concept} in 10 words.')
    | ChatOpenAI(model='gpt-4o-mini', temperature=0)
    | StrOutputParser()
)

# ── invoke: single input, single output ──
result = chain.invoke({'concept': 'recursion'})
print('invoke():', result)

# ── stream: yields chunks as they arrive ──
print('\nstream():', end=' ')
for chunk in chain.stream({'concept': 'neural networks'}):
    print(chunk, end='', flush=True)

# ── batch: multiple inputs, processed efficiently ──
results = chain.batch([
    {'concept': 'Python'},
    {'concept': 'LangChain'},
    {'concept': 'embeddings'},
])
print('\n\nbatch():')
for i, r in enumerate(results):
    print(f'  [{i}]: {r}')

In [ ]:
# ── RunnablePassthrough: Pass Input Through Unchanged ─────────────────
# WHY: Sometimes you want to pass the original input ALONGSIDE transformed data.
# Example: pass user question AND retrieved docs to the LLM.

from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# RunnableParallel runs multiple chains SIMULTANEOUSLY
# RunnablePassthrough passes input unchanged

parallel_chain = RunnableParallel({
    'original_input': RunnablePassthrough(),  # passes input as-is
    'uppercased': lambda x: x['topic'].upper(),  # transforms input
})

result = parallel_chain.invoke({'topic': 'langchain'})
print('Parallel result:', result)
# → {'original_input': {'topic': 'langchain'}, 'uppercased': 'LANGCHAIN'}

In [ ]:
# ── RunnableLambda: Wrap Any Python Function ────────────────────────────
# WHY: Sometimes you need custom logic inside a chain.
# RunnableLambda wraps any Python function into a Runnable.

from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

def add_context(inputs: dict) -> dict:
    """Custom preprocessing step — add date to prompt context."""
    from datetime import date
    inputs['date'] = date.today().isoformat()
    return inputs

from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate.from_template(
    'Today is {date}. Answer this question: {question}'
)
chain = RunnableLambda(add_context) | template | ChatOpenAI(model='gpt-4o-mini') | StrOutputParser()

result = chain.invoke({'question': 'What day is it?'})
print('Result:', result)

## 🎯 Interview Questions

1. **What does the `|` operator do in LangChain LCEL?**
2. **What is RunnableSequence?**
3. **Difference between invoke(), stream(), and batch()?**
4. **What is RunnablePassthrough used for?**
5. **How do you add custom Python logic inside an LCEL chain?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **LCEL — LangChain Expression Language**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
